# 03 | Desemprego estadual (BLS)

Preparamos a taxa anual de desemprego para comparar estados com condições econômicas diferentes. O notebook aceita a planilha bruta do BLS ou, nesta máquina, o CSV processado já gerado anteriormente.

## Bloco 1: Abrir

Se o bruto existir, lemos `data/raw/staadata/staadata.xlsx`. Caso contrário, usamos `data/processed/bls_unemployment_2019_2024.csv` e avisamos na tela.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
bruto = ROOT / "data/raw/staadata/staadata.xlsx"
processado = ROOT / "data/processed/bls_unemployment_2019_2024.csv"
if bruto.exists():
    bls = pd.read_excel(bruto, sheet_name=0, header=None, skiprows=8, usecols=[0, 1, 2, 9], names=["fips", "state", "year", "unemployment_rate"])
    origem = "bruto"
elif processado.exists():
    bls = pd.read_csv(processado)
    origem = "processado"
else:
    raise FileNotFoundError("Baixe staadata.xlsx do BLS e coloque em data/raw/staadata/.")
print("Origem:", origem, "| Linhas lidas:", len(bls))

## Bloco 2: Selecionar estados e anos

Usamos os mesmos 50 estados do CDC e os anos de 2019 a 2024.

In [ ]:
cdc = pd.read_csv(ROOT / "data/raw/cdc_homicides_2019_2024.csv")
estados = set(cdc["name"]) - {"District of Columbia"}
bls["year"] = pd.to_numeric(bls["year"], errors="coerce")
limpo = bls[bls["state"].isin(estados) & bls["year"].between(2019, 2024)][["state", "year", "unemployment_rate"]].copy()
limpo["year"] = limpo["year"].astype(int)
limpo["unemployment_rate"] = pd.to_numeric(limpo["unemployment_rate"], errors="coerce")
print("Dimensões:", limpo.shape)
display(limpo.head())

## Bloco 3: Validar e salvar

Esperamos 300 estado-anos, sem taxas vazias ou fora do intervalo 0 a 100%.

In [ ]:
assert limpo.shape == (300, 3)
assert not limpo.duplicated(["state", "year"]).any()
assert limpo["unemployment_rate"].notna().all()
assert limpo["unemployment_rate"].between(0, 100).all()
processado.parent.mkdir(parents=True, exist_ok=True)
limpo.to_csv(processado, index=False)
print("OK, salvo:", processado.relative_to(ROOT))